# Climate quickstart

The shortest run through the committed data, for the Philippines; the full version is `degree_days_from_daily_grid.ipynb`.

Read one country's daily highs and lows.

In [1]:
import logging
import sys

import numpy as np
import pandas as pd

import cil_regionalization as cilreg
from cil_regionalization.bcsd_io import BCSD_GRID, read_bcsd_daily

sys.path.insert(0, ".")
import data

tasmax = read_bcsd_daily(str(data.daily_path("tasmax", "PHL")), "tasmax",
                         require_full_year=True)
tasmin = read_bcsd_daily(str(data.daily_path("tasmin", "PHL")), "tasmin",
                         require_full_year=True)
print(f"{data.days_in_file('tasmax', 'PHL')} days in the file, "
      f"{len(tasmax.dates)} returned, values {tasmax.values.shape} (time, lat, lon)")

366 days in the file, 365 returned, values (365, 67, 40) (time, lat, lon)


Cooling degree days above 20 C and days above 35 C, per cell.

In [2]:
tas = (tasmax.values.astype("float64") + tasmin.values) / 2.0 - 273.15
cdd = np.maximum(tas - 20.0, 0.0).sum(axis=0)
hot = (tasmax.values.astype("float64") - 273.15 > 35.0).sum(axis=0).astype("float64")
hot[np.isnan(tasmax.values).any(axis=0)] = np.nan   # a blank day leaves the cell blank
print(f"cdd and hot_days {cdd.shape} (lat, lon), {int(np.isfinite(cdd).sum())} cells with data")

cdd and hot_days (67, 40) (lat, lon), 817 cells with data


Turn the grid into a table of cells.

In [3]:
cells = tasmax.to_cells({"cdd": cdd, "hot_days": hot})
print(f"{len(cells)} cells")
print(cells.head().round(1).to_string(index=False))

817 cells
 cell_ix  cell_iy    cdd  hot_days
    1197      378 2705.3       0.0
    1198      378 2602.3       0.0
    1197      379 2727.5       0.0
    1198      379 2698.0       0.0
    1199      379 2596.3       0.0


Read the Philippines' impact region segment weights with `from_table`.

In [4]:
table = pd.read_parquet(data.IR_TABLE)
table = table[table["hierid"].str[:3] == "PHL"]
weights = cilreg.WeightsArtifact.from_table(
    table,
    id_fields=["hierid"],
    weights={"pop": "popwt", "area": "areawt"},
    source=BCSD_GRID,
)
print(f"{len(weights.frame)} rows, {weights.frame['hierid'].nunique()} impact regions")

1462 rows, 200 impact regions


Average both fields over each impact region, weighted by population, in one call.

In [5]:
# the skipped cells are summarised below instead of logged one case at a time
logging.getLogger("cil_regionalization.validate").setLevel(logging.ERROR)
policies = cilreg.SourceUnitPolicies(on_zero_weight="skip", on_unmatched="skip")
long = cells.melt(id_vars=["cell_ix", "cell_iy"], var_name="field")
result = cilreg.apply_weights(
    weights,
    long,
    kind="intensive",
    weight="pop",
    data_version=BCSD_GRID,
    value_col="value",
    policies=policies,
)
skipped = result.coverage.counts
print(f"{result.frame['hierid'].nunique()} impact regions; skipped {skipped['unmatched']} cells "
      f"the weights do not cover and {skipped['zero_weight']} with no population")

200 impact regions; skipped 13 cells the weights do not cover and 42 with no population


The first rows of the result.

In [6]:
print(result.frame.pivot(index="hierid", columns="field", values="value")
      .rename_axis(columns=None).reset_index()
      .rename(columns={"cdd": "cdd (C x days above 20)", "hot_days": "hot_days (days above 35 C)"})
      .head().round(1).to_string(index=False))

                      hierid  cdd (C x days above 20)  hot_days (days above 35 C)
                       PHL.1                   2152.3                         0.0
                      PHL.10                   2376.5                         0.1
PHL.12.159.R1098fc7ffd828584                   2297.1                         0.0
PHL.12.159.R63194439249382e3                   2270.5                         0.0
PHL.12.169.R8c8079d4424a6183                   2251.0                         0.0
